In [1]:
import asyncio, json,  ssl, websockets, uuid
from pprint import pprint

In [11]:
TV_IP="192.168.0.122"
TV_MAC="90:E4:68:53:4E:09"
TV_CLIENT_KEY= "616a947803abca59bf5162d54fe68532",
# "1b07f1426afc68c9df23843f71e74fcb"
# old_key = "6071d1f3d832eef74a8df52fcf0d6b8b"

In [20]:
async def connect_to_tv(ip_address, client_key):
    uri = f"wss://{ip_address}:3001/"
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE
    
    try:
        websocket = await websockets.connect(uri, ssl=ssl_context)
        
        # Register with client key (expanded permissions)
        register_payload = {
            "type": "register",
            "id": "register_0",
            "payload": {
                "forcePairing": False,  # Keep True for now to force re-pairing; set to False after
                "pairingType": "PROMPT",
                "client-key": client_key,
                "manifest": {
                    "manifestVersion": 1,
                    "permissions": [
                        "LAUNCH", "LAUNCH_WEBAPP", "APP_TO_APP", "CLOSE", "TEST_OPEN", "TEST_PROTECTED",
                        "CONTROL_AUDIO", "CONTROL_DISPLAY", "CONTROL_INPUT_JOYSTICK",
                        "CONTROL_INPUT_MEDIA_RECORDING", "CONTROL_INPUT_MEDIA_PLAYBACK", "CONTROL_INPUT_TV",
                        "CONTROL_POWER", "READ_APP_STATUS", "READ_CURRENT_CHANNEL", "READ_INPUT_DEVICE_LIST",
                        "READ_NETWORK_STATE", "READ_RUNNING_APPS", "READ_TV_CHANNEL_LIST", "WRITE_NOTIFICATION_TOAST",
                        "READ_POWER_STATE", "READ_COUNTRY_INFO", "READ_SETTINGS", "CONTROL_MOUSE_AND_KEYBOARD",
                        "CONTROL_INPUT_TEXT", "CONTROL_USER_INFO", "READ_UPDATE_INFO",
                        "UPDATE_FROM_REMOTE_APP", "READ_LGE_TV_INPUT_EVENTS", "READ_TV_CURRENT_TIME","CONTROL_AUDIO_OUTPUT","READ_AUDIO_DEVICES"
                    ],
                    "signatures": [{"signatureVersion": 1}],
                    "appVersion": "1.0"
                }
            }
        }
        await websocket.send(json.dumps(register_payload))
        
        # Loop to receive responses until "registered" or timeout
        start_time = asyncio.get_event_loop().time()
        while True:
            try:
                response = await asyncio.wait_for(websocket.recv(), timeout=30)
                reg_response = json.loads(response)
                # print(f"Received response: {reg_response}")  # Debug print
                
                # Extract client key if present
                new_key = reg_response.get('payload', {}).get('client-key')
                if new_key:
                    # print(f"New client key from TV: {new_key}  # Save and use this for future connections!")
                    client_key = new_key  # Update the client key for future use
                if reg_response.get("type") == "registered":
                    # print("Registration successful.")
                    return websocket
                elif reg_response.get("type") == "error":
                    print(f"Registration error: {reg_response}")
                    await websocket.close()
                    return None
            except asyncio.TimeoutError:
                print("Timeout waiting for registration (did you accept the TV prompt?)")
                await websocket.close()
                return None
    
    except Exception as e:
        print(f"Connection error (TV might be off): {str(e)}")
        return None

In [21]:
async def send_command(websocket, uri, payload=None):
    if payload is None:
        payload = {}
    
    request_id = f"req_{uuid.uuid4().hex[:8]}"  # Unique ID
    command = {
        "type": "request",
        "id": request_id,
        "uri": uri,
        "payload": payload
    }
    await websocket.send(json.dumps(command))
    
    # Await response (assuming single response; in production, might need to loop/handle multiple)
    response = await websocket.recv()
    resp_json = json.loads(response)
    
    if resp_json.get("type") == "response" and resp_json.get("id") == request_id:
        return resp_json.get("payload", {})
    else:
        return {"error": f"Unexpected response: {resp_json}"}

In [22]:
async def close_connection(websocket):
    if websocket:
        await websocket.close()
        print("Connection closed.")

In [23]:
await connect_to_tv(TV_IP, TV_CLIENT_KEY)

In [ ]:
common_audio_outputs = [
    {"name": "tv_speaker", "description": "Internal TV speakers"},
    {"name": "external_arc", "description": "HDMI-ARC/eARC (for soundbars/AV receivers)"},
    {"name": "external_optical", "description": "Optical/Toslink output"},
    {"name": "lineout", "description": "Analog line out"},
    {"name": "headphone", "description": "Wired headphones"},
    {"name": "bt_soundbar", "description": "Bluetooth soundbar or speaker"},
    {"name": "tv_speaker_bluetooth", "description": "TV speakers + Bluetooth device"},
    {"name": "tv_external_speaker", "description": "TV speakers + external speaker"},
    {"name": "tv_speaker_headphone", "description": "TV speakers + headphones"}
]

In [47]:
ws = await connect_to_tv(TV_IP, TV_CLIENT_KEY)

if ws:
    
    # Get current sound output
    current_output_resp = await send_command(ws, "ssap://com.webos.service.apiadapter/audio/getSoundOutput")
    current_output = current_output_resp.get('soundOutput', 'Unknown')
    print(f"Current Sound Output: {current_output}")
    
    # Determine the target output to toggle
    if current_output == "tv_speaker":
        set_output_speaker = "bt_soundbar"
    elif current_output == "bt_soundbar":
        set_output_speaker = "tv_speaker"
    else:
        set_output_speaker = "tv_speaker"  # Default to internal if unknown/other
        print(f"Unknown current output '{current_output}'—defaulting to toggle to {set_output_speaker}")
    
    # Switch to the target output
    set_output = await send_command(ws, "ssap://com.webos.service.apiadapter/audio/changeSoundOutput", {"output": set_output_speaker})
    print(f"Set Output Response: {set_output}")
    
    # Verify the new output (optional)
    new_output_resp = await send_command(ws, "ssap://com.webos.service.apiadapter/audio/getSoundOutput")
    new_output = new_output_resp.get('soundOutput', 'Unknown')
    print(f"New Sound Output: {new_output}")

Current Sound Output: tv_speaker
Set Output Response: {'returnValue': True, 'method': 'setSystemSettings'}
New Sound Output: bt_soundbar


In [ ]:
ws = await connect_to_tv(TV_IP, TV_CLIENT_KEY)

if ws:
    current_output = await send_command(ws, "ssap://com.webos.service.apiadapter/audio/getSoundOutput")
    print(f"Current Sound Output: {current_output.get('soundOutput', 'Unknown')}")
    if current_output.get('soundOutput', 'Unknown') == "tv_speaker":
        set_output_speaker = "bt_soundbar"
    if current_output.get('soundOutput', 'Unknown') == "bt_soundbar":
        set_output_speaker = "tv_speaker"
    # Example: Switch to a different output (e.g., 'tv_speaker')
    set_output = await send_command(ws, "ssap://com.webos.service.apiadapter/audio/changeSoundOutput", {"output": set_output_speaker})
    print(f"Set Output Response: {set_output}")

Current Sound Output: bt_soundbar
Set Output Response: {'returnValue': True, 'method': 'setSystemSettings'}


In [30]:
ws = await connect_to_tv(TV_IP, TV_CLIENT_KEY)

if ws:
    
    audio_devices = await send_command(ws, "ssap://com.webos.service.audio/listSupportedDevices", {"query": "output"})
    print(f"Available Audio Devices (full payload): {audio_devices}")

    audio_devices = await send_command(ws, "ssap://audio/listSupportedDevices", {"query": "output"})
    print(f"Available Audio Devices (full payload): {audio_devices}")
    
    audio_devices = await send_command(ws, "ssap://audio/listSupportedDevices")
    print(f"Available Audio Devices (full payload): {audio_devices}")
   
    audio_devices = await send_command(ws, "ssap://com.webos.service.audiooutput/audio/volume/getStatus",{"query":"all"})
    print(f"Available Audio Devices (full payload): {audio_devices}")
    # If the above fails with 404 or 401, try this variant
    # audio_devices_alt = await send_command(ws, "ssap://audio/listSupportedDevices", {"query": "output"})
    # print(f"Alternative Audio Devices: {audio_devices_alt}")

    # Fallback: Hardcoded common audio outputs for LG WebOS TVs (based on your model and common integrations)
    common_audio_outputs = [
        {"name": "tv_speaker", "description": "Internal TV speakers"},
        {"name": "external_arc", "description": "HDMI-ARC/eARC (for soundbars/AV receivers)"},
        {"name": "external_optical", "description": "Optical/Toslink output"},
        {"name": "lineout", "description": "Analog line out"},
        {"name": "headphone", "description": "Wired headphones"},
        {"name": "bt_soundbar", "description": "Bluetooth soundbar or speaker"},
        {"name": "wisa_speaker", "description": "WiSA wireless speaker"},
        {"name": "lg_soundsync", "description": "LG Sound Sync (optical or Bluetooth)"},
        {"name": "tv_external_speaker", "description": "TV speakers + external speaker"},
        {"name": "tv_speaker_headphone", "description": "TV speakers + headphones"},
        {"name": "tv_speaker_bluetooth", "description": "TV speakers + Bluetooth device"}
    ]
    # print("Common Available Audio Outputs:")
    # for output in common_audio_outputs:
    #     print(f"- {output['name']}: {output['description']}")

    # Current sound output (for reference)
    current_output = await send_command(ws, "ssap://com.webos.service.apiadapter/audio/getSoundOutput",)
    print(f"Current Sound Output: {current_output.get('soundOutput', 'Unknown')}")

    # To switch (example to bt_soundbar if available)
    # set_output = await send_command(ws, "ssap://com.webos.service.apiadapter/audio/changeSoundOutput", {"output": "bt_soundbar"})
    # print(f"Set Output Response: {set_output}")

Available Audio Devices (full payload): {'error': "Unexpected response: {'type': 'error', 'id': 'req_bead3b94', 'error': '404 no such service or method', 'payload': {}}"}
Available Audio Devices (full payload): {'error': "Unexpected response: {'type': 'error', 'id': 'req_33780ce4', 'error': '404 no such service or method', 'payload': {}}"}
Available Audio Devices (full payload): {'error': "Unexpected response: {'type': 'error', 'id': 'req_eea63e5b', 'error': '404 no such service or method', 'payload': {}}"}
Available Audio Devices (full payload): {'error': "Unexpected response: {'type': 'error', 'id': 'req_9cf12abe', 'error': '404 no such service or method', 'payload': {}}"}
Current Sound Output: tv_speaker


In [24]:
ws = await connect_to_tv(TV_IP, TV_CLIENT_KEY)

if ws:
    audio_devices = await send_command(ws, "ssap://com.webos.service.audiooutput/getDeviceList")
    print(f"\nAvailable Audio Devices (full payload):")
    pprint(audio_devices)

    audio_status = await send_command(ws, "ssap://audio/getStatus")
    print(f"\nAudio Status (full payload")
    pprint(audio_status)
    # current_output = audio_status.get('soundOutput', 'Unknown')
    # print(f"Current Audio Output: {current_output}")

    # Example: Get current sound output directly
    sound_output = await send_command(ws, "ssap://com.webos.service.apiadapter/audio/getSoundOutput")
    print(f"\nCurrent Sound Output:")
    pprint(sound_output.get('soundOutput', 'Unknown'))


Available Audio Devices (full payload):
{'error': "Unexpected response: {'type': 'error', 'id': 'req_061de1ba', "
          "'error': '404 no such service or method', 'payload': {}}"}

Audio Status (full payload
{'callerId': 'com.webos.service.apiadapter',
 'mute': False,
 'returnValue': True,
 'volume': 9,
 'volumeStatus': {'activeStatus': True,
                  'adjustVolume': True,
                  'externalDeviceControl': False,
                  'maxVolume': 100,
                  'mode': 'normal',
                  'muteStatus': False,
                  'soundOutput': 'tv_speaker',
                  'volume': 9,
                  'volumeLimitable': True,
                  'volumeLimiter': 'none',
                  'volumeSyncable': True}}

Current Sound Output:
'tv_speaker'


In [1]:
ws = await connect_to_tv(TV_IP, TV_CLIENT_KEY)

if ws:
    # Example: Get power state
    power_state = await send_command(ws, "ssap://com.webos.service.tvpower/power/getPowerState")
    print(f"TV Power State: {power_state.get('state', 'Unknown')}")

    # system_info = await send_command(ws, "ssap://system/getSystemInfo")
    # print(f"System Info: {system_info}")

    audio_outputs = await send_command(ws, "ssap://com.webos.service.audiooutput/audio/volume/getStatus")
    print(f"Available Audio Outputs: {audio_outputs.get('volumeStatus', 'Unknown')}")

    supported_devices = await send_command(ws, "ssap://com.webos.service.audio/listSupportedDevices")
    print(f"Supported Audio Devices: {supported_devices.get('devices', 'Unknown')}")

    # Close when done
    await close_connection(ws)

NameError: name 'connect_to_tv' is not defined

ConnectionClosedOK: sent 1000 (OK); then received 1000 (OK)